# Experiment 10: LLM-Refined Q-Matrix

This notebook implements a process to refine Knowledge Component (KC) tags for Java programming problems. For each problem, we analyze actual student code (passing vs. failing) using a Large Language Model (Gemini 2.0 Flash) to determine which KCs are actually being tested.

The core idea is:
1. **Identify Genuine attempts**: Filter out trivial code.
2. **Contextualize failures**: Show the LLM what passing students do right and failing students get wrong.
3. **Refine Mapping**: The LLM updates the instructor's KC tags to reflect what differentiates performance.
4. **Evaluate**: The refined Q-matrix is then used in a Performance Factor Analysis (PFA) pipeline.

In [1]:
import pandas as pd
import numpy as np
import json
import time
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

# === CONFIGURATION ===
PROJECT_ROOT = Path("/mnt/d/Projects/kintsugi")
DATA_DIR = PROJECT_ROOT / "dataset" / "CodeWorkout"
RESULTS_DIR = PROJECT_ROOT / "results" / "10_qmatrix_refinement"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
MODEL_ID = "gemini-2.0-flash"  # Using the new SDK with this model
SLEEP_SECONDS = 1.5
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

# Initialize Gemini client
if not GEMINI_API_KEY:
    print("WARNING: GEMINI_API_KEY not found in environment. Please set it in your .env file.")
    client = None
else:
    client = genai.Client(api_key=GEMINI_API_KEY)
    print(f"Gemini client initialized with model {MODEL_ID}")

Gemini client initialized with model gemini-2.0-flash


In [2]:
# === LOAD DATA ===
print("Loading data...")

# Paths to data
MAINTABLE_PATH = DATA_DIR / "MainTable.csv"
SUBJECT_TABLE_PATH = DATA_DIR / "LinkTables" / "Subject.csv"
CODESTATES_TABLE_PATH = DATA_DIR / "LinkTables" / "CodeStates.csv"
PROBLEM_PROMPT_PATH = DATA_DIR / "Problem_Prompts" / "problem_prompts.csv"

# Load the full dataset
main_df = pd.read_csv(MAINTABLE_PATH)
subject_df = pd.read_csv(SUBJECT_TABLE_PATH)
codestates_df = pd.read_csv(CODESTATES_TABLE_PATH)
pp_df = pd.read_csv(PROBLEM_PROMPT_PATH)

# Merge to get subject info and code
# Filter MainTable to Run.Program events only (these have test scores)
df = main_df[main_df["EventType"] == "Run.Program"].merge(
    subject_df, on="SubjectID"
).merge(
    codestates_df, on="CodeStateID"
)

# Get best attempts (highest score per student per problem)
best = df.sort_values("Score", ascending=False).drop_duplicates(
    subset=["SubjectID", "ProblemID"], keep="first"
)

# Binarize: 1.0 = pass, else fail
best["passed"] = (best["Score"] == 1.0).astype(int)

# KC columns - Everything after 'Requirement'
KC_COLS = list(pp_df.columns[3:])
ALL_KCS = KC_COLS

print(f"Dataset: {len(best)} best attempts across {best['SubjectID'].nunique()} students")
print(f"Problems in Q-matrix: {pp_df['ProblemID'].nunique()}")
print(f"Knowledge Components (18): {len(ALL_KCS)}")

Loading data...
Dataset: 15375 best attempts across 372 students
Problems in Q-matrix: 50
Knowledge Components (18): 18


In [3]:
# === STEP 2: SMART SAMPLING ===

MIN_CODE_LENGTH = 50  # Characters — filters out trivial "return X" submissions
MIN_CODE_LINES = 3    # Lines — must have attempted some structure

def is_genuine_attempt(code_str):
    """Filter out garbage submissions that are just return statements."""
    if pd.isna(code_str):
        return False
    code = str(code_str).strip()
    if len(code) < MIN_CODE_LENGTH:
        return False
    if code.count('\n') + 1 < MIN_CODE_LINES:
        return False
    return True

def get_problem_difficulty(problem_id, best_df):
    """Compute difficulty metrics for a problem from class-level data."""
    prob_data = best_df[best_df["ProblemID"] == problem_id]
    total = len(prob_data)
    if total == 0:
        return {"pass_rate": 0, "avg_score": 0, "total_attempts": 0, "difficulty": "unknown"}
    
    pass_rate = prob_data["passed"].mean()
    avg_score = prob_data["Score"].mean()
    
    if pass_rate >= 0.8:
        difficulty = "easy"
    elif pass_rate >= 0.5:
        difficulty = "medium"
    else:
        difficulty = "hard"
    
    return {
        "pass_rate": round(pass_rate, 3),
        "avg_score": round(avg_score, 3),
        "total_attempts": total,
        "difficulty": difficulty,
    }

def get_code_samples(problem_id, best_df, n_pass=5, n_fail=5):
    """
    Get n passing and n failing code samples for a problem.
    """
    prob_data = best_df[best_df["ProblemID"] == problem_id].copy()
    
    # Filter to genuine attempts only
    prob_data = prob_data[prob_data["Code"].apply(is_genuine_attempt)]
    
    passing = prob_data[prob_data["passed"] == 1]
    failing = prob_data[prob_data["passed"] == 0]
    
    # For failing: prefer partial scores (more informative than zero scores)
    partial_fails = failing[failing["Score"] > 0].sort_values("Score", ascending=False)
    zero_fails = failing[failing["Score"] == 0]
    
    # Take partial failures first, then fill with zero-score genuine attempts
    if len(partial_fails) >= n_fail:
        fail_sample = partial_fails.sample(n=n_fail, random_state=RANDOM_SEED)
    else:
        # Take all partial + fill remainder from zero scores
        remaining = n_fail - len(partial_fails)
        if len(zero_fails) >= remaining:
            zero_sample = zero_fails.sample(n=remaining, random_state=RANDOM_SEED)
        else:
            zero_sample = zero_fails
        fail_sample = pd.concat([partial_fails, zero_sample])
    
    # For passing: just sample randomly from genuine attempts
    if len(passing) >= n_pass:
        pass_sample = passing.sample(n=n_pass, random_state=RANDOM_SEED)
    else:
        pass_sample = passing
    
    return pass_sample, fail_sample

In [4]:
# === STEP 3: THE PROMPT ===

def build_qmatrix_prompt(problem_id, requirement, instructor_kcs,
                          passing_codes, passing_scores,
                          failing_codes, failing_scores,
                          difficulty_info):
    """Build the prompt for Q-matrix refinement."""
    
    # Format passing code samples with scores
    pass_section = ""
    for i, (code, score) in enumerate(zip(passing_codes, passing_scores), 1):
        pass_section += f"\n--- Passing Student {i} (Score: {score:.2f} = all test cases correct) ---\n{code}\n"
    
    # Format failing code samples with scores
    fail_section = ""
    for i, (code, score) in enumerate(zip(failing_codes, failing_scores), 1):
        pct = int(score * 100)
        label = f"{pct}% of test cases passed" if score > 0 else "0% — failed all test cases"
        fail_section += f"\n--- Failing Student {i} (Score: {score:.2f} = {label}) ---\n{code}\n"
    
    prompt = f"""You are an expert CS education researcher analyzing a Java programming problem to determine which Knowledge Components (KCs) it ACTUALLY tests in practice.

PROBLEM (ID: {problem_id}):
{requirement}

PROBLEM DIFFICULTY:
- Class pass rate: {difficulty_info['pass_rate']*100:.1f}% of students scored 100%
- Class average score: {difficulty_info['avg_score']*100:.1f}%
- Total students who attempted: {difficulty_info['total_attempts']}
- Difficulty level: {difficulty_info['difficulty']}

INSTRUCTOR'S KC TAGS FOR THIS PROBLEM:
{json.dumps(instructor_kcs)}

THE 18 POSSIBLE KCs ARE:
{json.dumps(ALL_KCS)}

Below are real student code submissions. All submissions shown are GENUINE ATTEMPTS where students wrote meaningful code (not just a return statement). Each student's score indicates what percentage of automated test cases they passed.

=== PASSING STUDENT CODE (scored 100% — all test cases correct) ===
{pass_section}

=== FAILING STUDENT CODE (scored below 100%) ===
{fail_section}

YOUR TASK:
Analyze what passing students do correctly that failing students get wrong. Based on this analysis, determine which of the 18 KCs this problem ACTUALLY tests — meaning which KCs are the CAUSE of student failures.

IMPORTANT GUIDELINES:
1. A KC is "actually tested" if getting it wrong is what causes students to FAIL. Look at the specific errors and differences between passing and failing code.
2. A KC that appears in every correct solution but is NOT what differentiates pass from fail should be REMOVED.
3. Pay attention to partial scores. A student who scores 70% has most things right but one specific KC wrong.
4. For hard problems (low pass rate), the tested KCs are likely the more advanced ones.
5. Only include KCs from the list of 18. Do not invent new KC names.
6. You may ADD a KC that the instructor did not tag if failing students clearly struggle with it.
7. You may REMOVE a KC that the instructor tagged if failing students handle that skill correctly and fail for other reasons.

RESPOND WITH ONLY a JSON object in this exact format, no other text:
{{
    "problem_id": {problem_id},
    "refined_kcs": ["KC1", "KC2", ...],
    "removed_kcs": ["KC_removed1", ...],
    "added_kcs": ["KC_added1", ...],
    "reasoning": "Brief explanation of what specifically differentiates passing from failing students in their code"
}}"""
    
    return prompt

## Step 3b: Preview what the LLM will see

This cell lets you inspect the data for any problem before sending it to the LLM. 
Browse through a few problems to sanity-check the inputs.

In [5]:
from IPython.display import display, HTML, Markdown

def preview_problem(problem_id, best_df, pp_df):
    """Display everything the LLM will see for a given problem."""
    
    row = pp_df[pp_df["ProblemID"] == problem_id].iloc[0]
    requirement = row["Requirement"]
    instructor_kcs = [kc for kc in KC_COLS if row[kc] == 1]
    
    pass_sample, fail_sample = get_code_samples(problem_id, best_df)
    difficulty = get_problem_difficulty(problem_id, best_df)
    
    # Count how many genuine vs garbage submissions exist
    all_prob = best_df[best_df["ProblemID"] == problem_id]
    all_failing = all_prob[all_prob["passed"] == 0]
    genuine_failing = all_failing[all_failing["Code"].apply(is_genuine_attempt)]
    garbage_failing = len(all_failing) - len(genuine_failing)
    
    print("=" * 80)
    print(f"PROBLEM {problem_id}")
    print("=" * 80)
    
    print(f"\nDescription:\n  {requirement[:200]}{'...' if len(requirement) > 200 else ''}")
    
    print(f"\nDifficulty:")
    print(f"  Pass rate: {difficulty['pass_rate']*100:.1f}%")
    print(f"  Avg score: {difficulty['avg_score']*100:.1f}%")
    print(f"  Total attempts: {difficulty['total_attempts']}")
    print(f"  Level: {difficulty['difficulty']}")
    
    print(f"\nInstructor KC tags ({len(instructor_kcs)}):")
    print(f"  {instructor_kcs}")
    
    print(f"\nSubmission stats:")
    print(f"  Passing (score=1.0): {len(all_prob[all_prob['passed']==1])}")
    print(f"  Failing total: {len(all_failing)}")
    print(f"  Failing genuine attempts: {len(genuine_failing)}")
    print(f"  Failing garbage filtered out: {garbage_failing}")
    
    print(f"\nSampled for LLM: {len(pass_sample)} passing, {len(fail_sample)} failing")
    
    # Show passing code
    print(f"\n{'─' * 40}")
    print("PASSING STUDENT CODE")
    print(f"{'─' * 40}")
    for i, (_, r) in enumerate(pass_sample.iterrows(), 1):
        print(f"\n--- Pass {i} | Student {r['SubjectID']} | Score: {r['Score']:.2f} ---")
        code = str(r["Code"])
        lines = code.split('\n')
        for line in lines[:20]:
            print(f"  {line}")
        if len(lines) > 20:
            print(f"  ... ({len(lines) - 20} more lines)")
    
    # Show failing code
    print(f"\n{'─' * 40}")
    print("FAILING STUDENT CODE")
    print(f"{'─' * 40}")
    if len(fail_sample) == 0:
        print("  (No genuine failing submissions — instructor tags will be kept as-is)")
    else:
        for i, (_, r) in enumerate(fail_sample.iterrows(), 1):
            pct = int(r["Score"] * 100)
            print(f"\n--- Fail {i} | Student {r['SubjectID']} | Score: {r['Score']:.2f} ({pct}% test cases passed) ---")
            code = str(r["Code"])
            lines = code.split('\n')
            for line in lines[:20]:
                print(f"  {line}")
            if len(lines) > 20:
                print(f"  ... ({len(lines) - 20} more lines)")
    
    print(f"\n{'=' * 80}\n")

# PREVIEW INDIVIDUAL PROBLEM
preview_problem(32, best, pp_df)

PROBLEM 32

Description:
  Write a function in Java that implements the following logic: Given a string str and a non-empty word, return a version of the original string where all chars have been replaced by pluses (+), except ...

Difficulty:
  Pass rate: 84.4%
  Avg score: 89.0%
  Total attempts: 315
  Level: easy

Instructor KC tags (8):
  ['If/Else', 'While', 'LogicCompareNum', 'StringFormat', 'StringConcat', 'StringIndex', 'StringLen', 'StringEqual']

Submission stats:
  Passing (score=1.0): 266
  Failing total: 49
  Failing genuine attempts: 49
  Failing garbage filtered out: 0

Sampled for LLM: 5 passing, 5 failing

────────────────────────────────────────
PASSING STUDENT CODE
────────────────────────────────────────

--- Pass 1 | Student 13451 | Score: 1.00 ---
  public String plusOut(String str, String word)
  {
  	String result = "";
      
      int position = 0;
      
  	while(position < str.length())
      {
  		if(str.substring(position).startsWith(word))
      	{
  			r

In [6]:
# === PREVIEW ALL 50 PROBLEMS (summary table) ===
summary_rows = []
for pid in sorted(pp_df["ProblemID"].unique()):
    row = pp_df[pp_df["ProblemID"] == pid].iloc[0]
    instructor_kcs = [kc for kc in KC_COLS if row[kc] == 1]
    
    all_prob = best[best["ProblemID"] == pid]
    all_failing = all_prob[all_prob["passed"] == 0]
    genuine_failing = all_failing[all_failing["Code"].apply(is_genuine_attempt)] if len(all_failing) > 0 else pd.DataFrame()
    
    difficulty = get_problem_difficulty(pid, best)
    
    ps, fs = get_code_samples(pid, best)
    
    summary_rows.append({
        "ProblemID": pid,
        "Difficulty": difficulty["difficulty"],
        "PassRate": f"{difficulty['pass_rate']*100:.0f}%",
        "AvgScore": f"{difficulty['avg_score']*100:.0f}%",
        "NumInstructorKCs": len(instructor_kcs),
        "FailingTotal": len(all_failing),
        "FailingGenuine": len(genuine_failing),
        "SampledPass": len(ps),
        "SampledFail": len(fs),
        "Ready": "YES" if len(fs) > 0 else "SKIP (no failures)",
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df[["ProblemID", "Difficulty", "PassRate", "NumInstructorKCs",
                     "FailingGenuine", "SampledPass", "SampledFail", "Ready"]])

,ProblemID,Difficulty,PassRate,NumInstructorKCs,FailingGenuine,SampledPass,SampledFail,Ready
0,1,easy,99%,4,3,5,3,YES
1,3,easy,96%,5,14,5,5,YES
2,5,easy,96%,3,12,5,5,YES
3,12,easy,99%,5,2,5,2,YES
4,13,easy,92%,6,27,5,5,YES
5,17,easy,99%,4,3,5,3,YES
6,20,easy,99%,5,4,5,4,YES
7,21,easy,98%,5,5,5,5,YES
8,22,easy,96%,6,13,5,5,YES
9,24,easy,93%,4,22,5,5,YES


In [7]:
# === STEP 4: RUN THE EXPERIMENT ===

results = []
errors = []

# Filter to get first 50 problems (from pp_df)
problem_ids = sorted(pp_df["ProblemID"].unique())
total = len(problem_ids)

print(f"Starting Q-matrix refinement for {total} problems...")

for idx, pid in enumerate(problem_ids):
    # Get problem info
    row = pp_df[pp_df["ProblemID"] == pid].iloc[0]
    requirement = row["Requirement"]
    instructor_kcs = [kc for kc in ALL_KCS if row[kc] == 1]
    
    # Get code samples
    pass_sample, fail_sample = get_code_samples(pid, best)
    passing_codes = pass_sample["Code"].tolist() if len(pass_sample) > 0 else []
    passing_scores = pass_sample["Score"].tolist() if len(pass_sample) > 0 else []
    failing_codes = fail_sample["Code"].tolist() if len(fail_sample) > 0 else []
    failing_scores = fail_sample["Score"].tolist() if len(fail_sample) > 0 else []
    
    # Get difficulty context
    difficulty_info = get_problem_difficulty(pid, best)
    
    # Skip if no genuine failing attempts exist
    if len(failing_codes) == 0:
        print(f"  [{idx+1}/{total}] Problem {pid}: SKIPPED (no genuine failures)")
        results.append({
            "ProblemID": pid,
            "Instructor_KCs": json.dumps(instructor_kcs),
            "Refined_KCs": json.dumps(instructor_kcs),  # Keep original
            "Removed_KCs": json.dumps([]),
            "Added_KCs": json.dumps([]),
            "Reasoning": "No genuine failing submissions found.",
            "Num_Instructor": len(instructor_kcs),
            "Num_Refined": len(instructor_kcs),
            "Num_Removed": 0,
            "Num_Added": 0,
            "Raw_Response": "SKIPPED",
        })
        continue
    
    # Build prompt
    prompt = build_qmatrix_prompt(
        pid, requirement, instructor_kcs,
        passing_codes, passing_scores,
        failing_codes, failing_scores,
        difficulty_info
    )
    
    # Call LLM
    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.2, # Low temperature for consistency
                max_output_tokens=1024,
            )
        )
        
        raw_text = response.text.strip()
        
        # Parse JSON
        clean = raw_text.replace("```json", "").replace("```", "").strip()
        parsed = json.loads(clean)
        
        refined_kcs = parsed.get("refined_kcs", [])
        removed = parsed.get("removed_kcs", [])
        added = parsed.get("added_kcs", [])
        reasoning = parsed.get("reasoning", "")
        
        results.append({
            "ProblemID": pid,
            "Instructor_KCs": json.dumps(instructor_kcs),
            "Refined_KCs": json.dumps(refined_kcs),
            "Removed_KCs": json.dumps(removed),
            "Added_KCs": json.dumps(added),
            "Reasoning": reasoning,
            "Num_Instructor": len(instructor_kcs),
            "Num_Refined": len(refined_kcs),
            "Num_Removed": len(removed),
            "Num_Added": len(added),
            "Raw_Response": raw_text,
        })
        
        print(f"  [{idx+1}/{total}] Problem {pid}: OK ({len(instructor_kcs)} -> {len(refined_kcs)} KCs)")
        
    except Exception as e:
        print(f"  [{idx+1}/{total}] Problem {pid}: ERROR! {str(e)}")
        errors.append({"ProblemID": pid, "Error": str(e)})
    
    time.sleep(SLEEP_SECONDS)

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_DIR / "qmatrix_refinement_results.csv", index=False)
if errors:
    pd.DataFrame(errors).to_csv(RESULTS_DIR / "qmatrix_errors.csv", index=False)

print(f"\nSaved {len(results_df)} results to {RESULTS_DIR}")

Starting Q-matrix refinement for 50 problems...
  [1/50] Problem 1: OK (4 -> 4 KCs)
  [2/50] Problem 3: OK (5 -> 4 KCs)
  [3/50] Problem 5: OK (3 -> 3 KCs)
  [4/50] Problem 12: OK (5 -> 4 KCs)
  [5/50] Problem 13: OK (6 -> 5 KCs)
  [6/50] Problem 17: OK (4 -> 3 KCs)
  [7/50] Problem 20: OK (5 -> 5 KCs)
  [8/50] Problem 21: OK (5 -> 3 KCs)
  [9/50] Problem 22: OK (6 -> 4 KCs)
  [10/50] Problem 24: OK (4 -> 4 KCs)
  [11/50] Problem 25: OK (4 -> 4 KCs)
  [12/50] Problem 28: OK (6 -> 4 KCs)
  [13/50] Problem 31: OK (5 -> 4 KCs)
  [14/50] Problem 32: OK (8 -> 5 KCs)
  [15/50] Problem 33: OK (6 -> 4 KCs)
  [16/50] Problem 34: OK (8 -> 6 KCs)
  [17/50] Problem 36: OK (8 -> 7 KCs)
  [18/50] Problem 37: OK (6 -> 5 KCs)
  [19/50] Problem 38: OK (7 -> 3 KCs)
  [20/50] Problem 39: OK (6 -> 4 KCs)
  [21/50] Problem 40: OK (7 -> 4 KCs)
  [22/50] Problem 41: OK (3 -> 2 KCs)
  [23/50] Problem 43: OK (4 -> 2 KCs)
  [24/50] Problem 44: OK (6 -> 5 KCs)
  [25/50] Problem 45: OK (6 -> 4 KCs)
  [26/50] Prob

In [8]:
# === STEP 5: BUILD THE REFINED Q-MATRIX ===

refined_qmatrix = []

for _, row in results_df.iterrows():
    pid = row["ProblemID"]
    refined_kcs = json.loads(row["Refined_KCs"])
    
    qrow = {"ProblemID": pid}
    for kc in ALL_KCS:
        qrow[kc] = 1 if kc in refined_kcs else 0
    refined_qmatrix.append(qrow)

refined_df = pd.DataFrame(refined_qmatrix)
refined_df.to_csv(RESULTS_DIR / "refined_problem_prompts.csv", index=False)

# Also save the mapping to the root as run3_kc_mapping.json for Experiment 09
mapping = {int(row["ProblemID"]): json.loads(row["Refined_KCs"]) for _, row in results_df.iterrows()}
with open(PROJECT_ROOT / "run3_kc_mapping.json", "w") as f:
    json.dump(mapping, f, indent=2)

print("=== Refined Q-Matrix Summary ===")
print(f"Problems: {len(refined_df)}")
for kc in ALL_KCS:
    orig = int(pp_df[kc].sum()) if kc in pp_df.columns else 0
    new = int(refined_df[kc].sum())
    delta = new - orig
    arrow = "+" if delta > 0 else ""
    print(f"  {kc:20s}: {orig:2d} -> {new:2d}  ({arrow}{delta})")
    
# Summary stats
print(f"\n=== Overall Changes ===")
print(f"Problems where KCs changed: {sum(1 for _, r in results_df.iterrows() if json.loads(r['Removed_KCs']) or json.loads(r['Added_KCs']))}/{len(results_df)}")
print(f"Average KCs per problem: Instructor={results_df['Num_Instructor'].mean():.1f}, Refined={results_df['Num_Refined'].mean():.1f}")
print(f"Total removals: {results_df['Num_Removed'].sum()}")
print(f"Total additions: {results_df['Num_Added'].sum()}")

=== Refined Q-Matrix Summary ===
Problems: 50
  If/Else             : 44 -> 30  (-14)
  NestedIf            : 10 ->  0  (-10)
  While               :  3 ->  1  (-2)
  For                 : 27 -> 24  (-3)
  NestedFor           :  4 ->  1  (-3)
  Math+-*/            : 20 -> 13  (-7)
  Math%               :  5 ->  6  (+1)
  LogicAndNotOr       : 30 -> 22  (-8)
  LogicCompareNum     : 39 -> 31  (-8)
  LogicBoolean        :  6 ->  7  (+1)
  StringFormat        : 14 ->  3  (-11)
  StringConcat        :  5 ->  5  (0)
  StringIndex         : 12 -> 12  (0)
  StringLen           :  9 -> 10  (+1)
  StringEqual         :  7 ->  6  (-1)
  CharEqual           :  4 ->  4  (0)
  ArrayIndex          : 20 -> 20  (0)
  DefFunction         :  2 ->  4  (+2)

=== Overall Changes ===
Problems where KCs changed: 39/50
Average KCs per problem: Instructor=5.2, Refined=4.0
Total removals: 68
Total additions: 1
